In [7]:
"""
UPV Formación Permanente Knowledge Base Extractor
=================================================

Entrada:
    JSON con URLs de formación permanente

Salida:
    carpeta UPV_FormacionPermanente_KB/*.md
"""


# ─────────────────────────────────────────────
# CELDA 1
# ─────────────────────────────────────────────

!pip install -q beautifulsoup4 markdownify requests


# ─────────────────────────────────────────────
# CELDA 2
# ─────────────────────────────────────────────

import os
import re
import json
import time
import pickle
import requests
import markdownify

from bs4 import BeautifulSoup
from google.colab import drive


drive.mount('/content/drive')


JSON_URL = (
    "/content/drive/MyDrive/TFG Teleco/JSONs/"
    "formacion_permanente_upv.json"
)


PATH_KB = (
    "/content/drive/MyDrive/TFG Teleco/"
    "UPV_FormacionPermanente_KB"
)


ESTADO = os.path.join(
    PATH_KB,
    "_estado.pkl"
)


PAUSA = 0.4


HEADERS = {
    "User-Agent":
    "Mozilla/5.0 (compatible; UPV-KB-Bot/2.0)"
}


os.makedirs(
    PATH_KB,
    exist_ok=True
)


# ─────────────────────────────────────────────
# CELDA 3
# ─────────────────────────────────────────────

def get(url):

    try:

        r = requests.get(
            url,
            headers=HEADERS,
            timeout=20
        )

        r.raise_for_status()

        time.sleep(PAUSA)

        return r.text

    except Exception as e:

        print(
            "Error:",
            url,
            e
        )

        return None



def cargar_estado():

    if os.path.exists(ESTADO):

        with open(
            ESTADO,
            "rb"
        ) as f:
            return pickle.load(f)

    return set()



def guardar_estado(s):

    with open(
        ESTADO,
        "wb"
    ) as f:
        pickle.dump(
            s,
            f
        )



# ─────────────────────────────────────────────
# CELDA 4
# extracción datos
# ─────────────────────────────────────────────


def buscar_patron(texto, patrones):

    for p in patrones:

        m = re.search(
            p,
            texto,
            re.I
        )

        if m:
            return m.group(1).strip()

    return None



def extraer_metadata_mejorada(texto):

    meta = {}


    meta["precio"] = buscar_patron(
        texto,
        [
            r"Precio\s+(\d+[.,]?\d*\s*€)",
            r"(\d+[.,]?\d*)\s*€"
        ]
    )


    # horas reales
    meta["horas"] = buscar_patron(
        texto,
        [
            r"Online\s+(\d+)\s*h",
            r"Presenciales\s+(\d+)\s*h",
            r"(\d+)\s*horas lectivas"
        ]
    )

    # creditos
    meta["ects"] = buscar_patron(
    texto,
    [
        r"ECTS\s*\n(\d+)",
        r"(\d+)\s*ECTS"
    ]
)

    # modalidad
    meta["modalidad"] = None

    for modalidad in [
        "Emisión en directo",
        "Semipresencial",
        "Online",
        "Presencial"
    ]:

        if re.search(
            r"\b" + re.escape(modalidad) + r"\b",
            texto,
            re.I
        ):
            meta["modalidad"] = modalidad
            break


    inicio = buscar_patron(
        texto,
        [
            r"Desde:\s*(.+)",
            r"Inicio:\s*(.+)"
        ]
    )


    fin = buscar_patron(
        texto,
        [
            r"Hasta:\s*(.+)",
            r"Fin:\s*(.+)"
        ]
    )


    if inicio and fin:
        meta["fechas"] = (
            inicio + " - " + fin
        )

    elif inicio:
        meta["fechas"] = inicio



    # campus
    meta["campus"] = buscar_patron(
        texto,
        [
            r"Campus\s+de\s+([A-Za-z]+)",
            r"Campus\s+([A-Za-z]+)"
        ]
    )


    # responsable
    meta["responsable"] = buscar_patron(
        texto,
        [
            r"Responsable de la actividad\s*(.+)",
            r"Responsable\s*\n(.+)",
            r"Director\s*\n(.+)"
        ]
    )



    # promotor
    meta["promotor"] = buscar_patron(
        texto,
        [
            r"Promovido por:\s*(.+)",
            r"Organiza:\s*(.+)",
            r"Departamento:\s*(.+)",
        ]
    )


    meta["dirigido_a"] = buscar_patron(
    texto,
    [
        r"Acción formativa dirigida a\s*(.*?)(?:\n\n|Objetivos)"
    ]
)

    meta["objetivos"] = buscar_patron(
    texto,
    [
        r"Objetivos\s*(.*?)(?:\n\n|Contenidos)"
    ]
)

    return meta



# ─────────────────────────────────────────────
# CELDA 5
# markdown
# ─────────────────────────────────────────────


# def html_a_markdown(html):

#     soup = BeautifulSoup(
#         html,
#         "html.parser"
#     )


#     for tag in soup(
#         [
#             "script",
#             "style",
#             "nav",
#             "footer",
#             "header"
#         ]
#     ):
#         tag.decompose()


#     main = (
#         soup.find("main")
#         or soup.body
#         or soup
#     )


#     md = markdownify.markdownify(
#         str(main),
#         heading_style="ATX",
#         strip=[
#             "img"
#         ]
#     )


#     return re.sub(
#         r"\n{3,}",
#         "\n\n",
#         md
#     ).strip()
def html_a_markdown_limpio(html):

    soup = BeautifulSoup(
        html,
        "html.parser"
    )


    # quitar basura global
    for tag in soup(
        [
            "script",
            "style",
            "nav",
            "footer",
            "header",
            "form",
            "button"
        ]
    ):
        tag.decompose()


    # eliminar bloques conocidos basura
    basura = [
        "Suscríbete al boletín",
        "Quiero recibir información",
        "Descarga en PDF",
        "Conoce la oferta de Formación Permanente",
        "También te puede interesar",
        "Buscar formación",
        "Iniciar sesión",
        "Registrarse",
        "Toggle navigation"
    ]


    for tag in soup.find_all(
        ["div","section","aside"]
    ):

        texto = tag.get_text(
            " ",
            strip=True
        )

        if any(
            b.lower() in texto.lower()
            for b in basura
        ):
            tag.decompose()



    main = (
        soup.find("main")
        or soup.body
        or soup
    )


    # eliminar enlaces de navegación
    for a in main.find_all("a"):

        a.replace_with(
            a.get_text(" ", strip=True)
        )


    md = markdownify.markdownify(
        str(main),
        heading_style="ATX"
    )


    # limpiar líneas inútiles

    patrones_eliminar = [
        r"Toggle navigation",
        r"Iniciar sesión",
        r"Registrarse",
        r"Buscar formación",
        r"ES\s*\|\s*VA",
        r"00 horas.*",
        r"Enviar",
        r"Nombre",
        r"Correo electrónico",
        r"He leído.*",
        r"© .*Universitat.*"
    ]


    for p in patrones_eliminar:

        md = re.sub(
            p,
            "",
            md,
            flags=re.I
        )


    return re.sub(
        r"\n{3,}",
        "\n\n",
        md
    ).strip()


def crear_markdown(item, html):


    soup = BeautifulSoup(
        html,
        "html.parser"
    )


    texto = soup.get_text(
        "\n",
        strip=True
    )


    meta = extraer_metadata_mejorada(
        texto
    )


    contenido = []


    contenido.append(
        f"# {item['nombre']}\n"
    )


    contenido.append(
        "## Información principal\n"
    )


    campos = [
        ("Precio", "precio"),
        ("Horas", "horas"),
        ("Modalidad", "modalidad"),
        ("Fechas", "fechas"),
        ("Campus", "campus"),
        ("Responsable", "responsable"),
        ("Promueve", "promotor")
    ]


    for nombre, clave in campos:

        if meta.get(clave):

            contenido.append(
                f"- **{nombre}:** {meta[clave]}"
            )


    contenido.append(
        ""
    )


    contenido.append(
        f"URL: {item['url']}"
    )


    contenido.append(
        "\n---\n"
    )


    # ahora sí limpiamos para contenido largo

    contenido.append(
        html_a_markdown_limpio(html)
    )


    return "\n".join(contenido)



# ─────────────────────────────────────────────
# CELDA 6
# proceso
# ─────────────────────────────────────────────


with open(
    JSON_URL,
    encoding="utf-8"
) as f:

    datos = json.load(f)



formaciones = datos["formaciones"]


procesados = cargar_estado()


pendientes = [
    f for f in formaciones
    if f["id"] not in procesados
]


print(
    "Pendientes:",
    len(pendientes)
)



for i,item in enumerate(
    pendientes,
    1
):

    print(
        f"[{i}/{len(pendientes)}]",
        item["nombre"]
    )


    html = get(
        item["url"]
    )


    if not html:
        continue



    md = crear_markdown(
        item,
        html
    )


    nombre = (
        re.sub(
            r"[^a-z0-9]+",
            "_",
            item["id"]
        )
        + ".md"
    )


    ruta = os.path.join(
        PATH_KB,
        nombre
    )


    with open(
        ruta,
        "w",
        encoding="utf-8"
    ) as f:

        f.write(md)



    procesados.add(
        item["id"]
    )


    if i % 10 == 0:
        guardar_estado(
            procesados
        )



guardar_estado(
    procesados
)


print(
    "✅ KB terminada:",
    PATH_KB
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Pendientes: 288
[1/288] ANÁLISIS DE LA COYUNTURA ECONÓMICA
[2/288] MODELOS MULTICRITERIO APLICADOS A LA GESTIÓN DE CARTERAS
[3/288] GESTIÓN DE CARTERAS II
[4/288] INCENDIOS DE ORIGEN ELÉCTRICO EN EL ÁMBITO DOMÉSTICO. CAUSAS, RIESGOS Y ACTUACIÓN
[5/288] CAMPOS MAGNÉTICOS EN INSTALACIONES ELÉCTRICAS Y ALREDEDORES, Y SU CÁLCULO Y REPRESENTACIÓN CON CRMAG PLUS
[6/288] ANÁLISIS Y DISEÑO DE PUESTAS A TIERRA EN INSTALACIONES ELÉCTRICAS CON CRGROUND®
[7/288] ASESOR FINANCIERO
[8/288] AGENTE FINANCIERO EUROPEO
[9/288] ASISTENTE FINANCIERO EUROPEO
[10/288] ACTUALIZACIÓN DE CONOCIMIENTOS EN ASESORÍA FINANCIERA 2025
[11/288] ACTUALIZACIÓN DE CONOCIMIENTOS EN CRÉDITO INMOBILIARIO 2025
[12/288] ASESOR FINANCIERO EN CRÉDITO HIPOTECARIO
[13/288] INFORMADOR FINANCIERO EN CRÉDITO HIPOTECARIO
[14/288] CLOUD COMPUTING CON AMAZON WEB SERVICES (AWS)
[15/288] BASES DE DATOS ESPACIA